# Task 3: Model Creation
Train models using 6 algorithms with sklearn library, plus one algorithm implemented from scratch.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv('cardio_cleaned.csv')
print(f"Dataset shape: {df.shape}")

X = df.drop('cardio', axis=1)
y = df['cardio']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, 'scaler.pkl')
print("Scaler saved.")

## 3.1 Linear Regression (as classifier)

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
lr_pred = (lr.predict(X_test_scaled) >= 0.5).astype(int)
lr_acc = accuracy_score(y_test, lr_pred)
print(f"Linear Regression Accuracy: {lr_acc:.4f}")
print(classification_report(y_test, lr_pred))
joblib.dump(lr, 'model_linear_regression.pkl')

## 3.2 Logistic Regression

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
log_pred = log_reg.predict(X_test_scaled)
log_acc = accuracy_score(y_test, log_pred)
print(f"Logistic Regression Accuracy: {log_acc:.4f}")
print(classification_report(y_test, log_pred))
joblib.dump(log_reg, 'model_logistic_regression.pkl')

## 3.3 Support Vector Machine (SVM)

In [ ]:
svm = SVC(kernel='rbf', random_state=42, probability=True)
svm.fit(X_train_scaled, y_train)
svm_pred = svm.predict(X_test_scaled)
svm_acc = accuracy_score(y_test, svm_pred)
print(f"SVM Accuracy: {svm_acc:.4f}")
print(classification_report(y_test, svm_pred))
joblib.dump(svm, 'model_svm.pkl')

## 3.4 K-Nearest Neighbors (KNN)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=11)
knn.fit(X_train_scaled, y_train)
knn_pred = knn.predict(X_test_scaled)
knn_acc = accuracy_score(y_test, knn_pred)
print(f"KNN Accuracy: {knn_acc:.4f}")
print(classification_report(y_test, knn_pred))
joblib.dump(knn, 'model_knn.pkl')

## 3.5 Decision Tree

In [ ]:
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X_train_scaled, y_train)
dt_pred = dt.predict(X_test_scaled)
dt_acc = accuracy_score(y_test, dt_pred)
print(f"Decision Tree Accuracy: {dt_acc:.4f}")
print(classification_report(y_test, dt_pred))
joblib.dump(dt, 'model_decision_tree.pkl')

## 3.6 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
rf_pred = rf.predict(X_test_scaled)
rf_acc = accuracy_score(y_test, rf_pred)
print(f"Random Forest Accuracy: {rf_acc:.4f}")
print(classification_report(y_test, rf_pred))
joblib.dump(rf, 'model_random_forest.pkl')

## 3.7 Logistic Regression from Scratch (Without Library)

In [ ]:
class LogisticRegressionScratch:
    def __init__(self, lr=0.01, n_iters=1000):
        self.lr = lr
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
        self.losses = []

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for i in range(self.n_iters):
            linear = np.dot(X, self.weights) + self.bias
            y_pred = self._sigmoid(linear)

            loss = -np.mean(y * np.log(y_pred + 1e-15) + (1 - y) * np.log(1 - y_pred + 1e-15))
            self.losses.append(loss)

            dw = (1 / n_samples) * np.dot(X.T, (y_pred - y))
            db = (1 / n_samples) * np.sum(y_pred - y)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

    def predict(self, X):
        linear = np.dot(X, self.weights) + self.bias
        y_pred = self._sigmoid(linear)
        return (y_pred >= 0.5).astype(int)


scratch_model = LogisticRegressionScratch(lr=0.1, n_iters=2000)
scratch_model.fit(X_train_scaled, y_train.values)
scratch_pred = scratch_model.predict(X_test_scaled)
scratch_acc = accuracy_score(y_test, scratch_pred)

print(f"Logistic Regression (Scratch) Accuracy: {scratch_acc:.4f}")
print(classification_report(y_test, scratch_pred))

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(scratch_model.losses, color='#e74c3c', linewidth=1.5)
plt.title('Logistic Regression (Scratch) — Training Loss', fontsize=14, fontweight='bold')
plt.xlabel('Iteration')
plt.ylabel('Binary Cross-Entropy Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
results = {
    'Linear Regression': lr_acc,
    'Logistic Regression': log_acc,
    'SVM': svm_acc,
    'KNN': knn_acc,
    'Decision Tree': dt_acc,
    'Random Forest': rf_acc,
    'LogReg (Scratch)': scratch_acc
}

results_df = pd.DataFrame(list(results.items()), columns=['Model', 'Accuracy'])
results_df = results_df.sort_values('Accuracy', ascending=False).reset_index(drop=True)
results_df['Accuracy'] = results_df['Accuracy'].round(4)
print(results_df.to_string(index=False))